In [1]:
# Conexión a BigQuery
# Importar módulos
import os
from dotenv import load_dotenv
from google.cloud import bigquery
from google.oauth2 import service_account

# Cargar variables de entorno
load_dotenv()
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Autenticación y cliente
credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH
)
client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

print(f"Conectado a BigQuery. Proyecto: {PROJECT_ID}")

Conectado a BigQuery. Proyecto: thebridge-ai-tc-sql


In [ ]:
# Ingresos por mes
query = f"""
    SELECT
        FORMAT_DATE('%Y-%m', DATE(payment_date)) AS sales_month, 
        COUNT(id) AS total_transactions, 
        ROUND(SUM(amount),2) AS total_revenue
    FROM `{PROJECT_ID}.{DATASET_ID}.payments`
    WHERE payment_status = 'completed'
    GROUP BY sales_month
    ORDER BY sales_month DESC
"""

# Ejecutar y obtener resultados
df_result = client.query(query).to_dataframe()
print(df_result)

   sales_month  total_transactions  total_revenue
0      2026-09                  76      123574.31
1      2026-08                 240      368429.66
2      2026-07                 164      269681.84
3      2026-06                 155      244780.95
4      2026-05                 154      239030.74
5      2026-04                 128      184154.93
6      2026-03                 128      208197.66
7      2026-02                  80      111972.74
8      2026-01                  86      116345.38
9      2025-12                  82      128221.85
10     2025-11                  77      112077.69
11     2025-10                  58       82495.76
12     2025-09                  46       78409.04
13     2025-08                  50       82003.05
14     2025-07                  37       59743.61
15     2025-06                  48       73463.61
16     2025-05                  37       59260.99
17     2025-04                  34       54629.99
18     2025-03                  23       28155.60


c:\Users\Rubén\Documents\01-Estudios\Bootcamp IA\00 - Repos\tc-sql-Ruben-Jimenez-Gutierrez\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [8]:
# Productos más vendidos
query = f"""
    SELECT
        p.product_name, 
        SUM(i.quantity) AS total_sold_units
    FROM `{PROJECT_ID}.{DATASET_ID}.order_items` i
    INNER JOIN `{PROJECT_ID}.{DATASET_ID}.products` p
        ON i.product_number = p.product_number
    GROUP BY p.product_name
    ORDER BY total_sold_units DESC
    LIMIT 10
"""

# Ejecutar y obtener resultados
df_result = client.query(query).to_dataframe()
print(df_result)

                            product_name  total_sold_units
0                  iPhone 15 Pro (Black)               107
1    Samsung Galaxy Watch Classic (64GB)               106
2        Dell XPS Gaming Edition (Black)               105
3     Apple Watch Series 9 Solar (512GB)               103
4  Apple Watch Series 9 Cellular (128GB)               102
5        Amazon Echo Dot Pack x2 (256GB)               102
6   Kindle Paperwhite Paper Edition (V2)                99
7           Amazon Echo Dot Color (64GB)                99
8            Fitbit Charge Active (64GB)                99
9            Cámara Ring Outdoor (128GB)                97


c:\Users\Rubén\Documents\01-Estudios\Bootcamp IA\00 - Repos\tc-sql-Ruben-Jimenez-Gutierrez\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
# Clientes por canal por el que conocieron la tienda

query = f"""
    SELECT 
        channel,
        COUNT(*) as total_customers
    FROM `{PROJECT_ID}.{DATASET_ID}.customers`
    GROUP BY channel
    ORDER BY total_customers DESC
"""

# Ejecutar y obtener resultados
df_result = client.query(query).to_dataframe()
print(df_result)

           channel  total_customers
0          organic              107
1      paid_search              106
2         referral              101
3     social_media               98
4  email_marketing               88


c:\Users\Rubén\Documents\01-Estudios\Bootcamp IA\00 - Repos\tc-sql-Ruben-Jimenez-Gutierrez\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [18]:
# Tiempo medio de entrega
query = f"""
    SELECT
        ROUND(AVG(DATE_DIFF(delivery_date, order_date, DAY)),1) AS average_delivery_time_in_days
    FROM `{PROJECT_ID}.{DATASET_ID}.orders`
    WHERE delivery_date IS NOT NULL
"""

# Ejecutar y obtener resultados
df_result = client.query(query).to_dataframe()
print(df_result)

   average_delivery_time_in_days
0                            4.6


c:\Users\Rubén\Documents\01-Estudios\Bootcamp IA\00 - Repos\tc-sql-Ruben-Jimenez-Gutierrez\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [24]:
# Rating medio y volumen de opiniones por categoría de producto

query = f"""
    SELECT
        c.category_name, 
        ROUND(AVG(r.rating), 1) AS average_rating, 
        COUNT(r.rating) AS total_reviews
    FROM `{PROJECT_ID}.{DATASET_ID}.categories` c
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.products` p
        ON c.id = p.category_id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.order_items` o
        ON p.product_number = o.product_number
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.reviews` r
        ON o.id = r.order_item_id
    WHERE r.rating IS NOT NULL
    GROUP BY c.category_name
    ORDER BY average_rating DESC, total_reviews DESC
"""

# Ejecutar y obtener resultados
df_result = client.query(query).to_dataframe()
print(df_result)

               category_name  average_rating  total_reviews
0                    Laptops             4.2            159
1                      Audio             4.2             85
2                  Wearables             4.1            229
3                 Smart Home             4.1            222
4        Tablets & E-readers             4.0            252
5  Componentes & Periféricos             4.0            113
6                Smartphones             4.0             70


c:\Users\Rubén\Documents\01-Estudios\Bootcamp IA\00 - Repos\tc-sql-Ruben-Jimenez-Gutierrez\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
